In [ ]:
!pip install spacy_transformers
!pip install -U spacy
!pip install -U spacy spacy-transformers tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 64.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.2
    Uninstalling transformers-4.55.2:
      Successfully uninstalled transformers-4.55.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 64.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import spacy
import json
from spacy.tokens import DocBin
from tqdm import tqdm
from sklearn.model_selection import train_test_split


In [ ]:
spacy.__version__

'3.8.7'

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!git clone https://github.com/laxmimerit/CV-Parsing-using-Spacy-3.git

Cloning into 'CV-Parsing-using-Spacy-3'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 82 (delta 14), reused 74 (delta 14), pack-reused 6 (from 1)
Receiving objects: 100% (82/82), 5.62 MiB | 23.38 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [ ]:
with open('/content/CV-Parsing-using-Spacy-3/data/training/train_data.json', 'r', encoding='utf-8') as f:
    cv_data = json.load(f)

In [ ]:
len(cv_data)

200

In [ ]:
!python -m spacy init fill-config \
    /content/CV-Parsing-using-Spacy-3/data/training/base_config.cfg \
    /content/CV-Parsing-using-Spacy-3/data/training/config.cfg

✔ Auto-filled config with all values
✔ Saved config
/content/CV-Parsing-using-Spacy-3/data/training/config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
cv_data[0]

['Govardhana K Senior Software Engineer  Bengaluru, Karnataka, Karnataka - Email me on Indeed: indeed.com/r/Govardhana-K/ b2de315d95905b68  Total IT experience 5 Years 6 Months Cloud Lending Solutions INC 4 Month • Salesforce Developer Oracle 5 Years 2 Month • Core Java Developer Languages Core Java, Go Lang Oracle PL-SQL programming, Sales Force Developer with APEX.  Designations & Promotions  Willing to relocate: Anywhere  WORK EXPERIENCE  Senior Software Engineer  Cloud Lending Solutions -  Bangalore, Karnataka -  January 2018 to Present  Present  Senior Consultant  Oracle -  Bangalore, Karnataka -  November 2016 to December 2017  Staff Consultant  Oracle -  Bangalore, Karnataka -  January 2014 to October 2016  Associate Consultant  Oracle -  Bangalore, Karnataka -  November 2012 to December 2013  EDUCATION  B.E in Computer Science Engineering  Adithya Institute of Technology -  Tamil Nadu  September 2008 to June 2012  https://www.indeed.com/r/Govardhana-K/b2de315d95905b68?isid=rex-

In [ ]:
def get_spacy_doc(file, data):
    nlp = spacy.blank("en")
    db = DocBin()

    for text, annot in tqdm(data):
        doc = nlp.make_doc(text)
        annot = annot["entities"]

        ents = []
        entity_indices = []

        for start, end, label in annot:
            # Skip invalid spans
            if start >= end or end > len(text):
                file.write(f"INVALID span: ({start}, {end}, {label}) in text: {text[:100]}\n")
                continue

            # Prevent overlap
            skip_entity = any(idx in entity_indices for idx in range(start, end))
            if skip_entity:
                continue

            entity_indices.extend(range(start, end))
            span = doc.char_span(start, end, label=label, alignment_mode="strict")

            if span is None:
                file.write(f"FAILED char_span: ({start}, {end}, {label}) in text: {text[:100]}\n")
            else:
                ents.append(span)

        doc.ents = ents
        db.add(doc)

    return db


In [ ]:
train, test = train_test_split(cv_data, test_size=0.3)

In [ ]:
len(train), len(test)

(140, 60)

In [ ]:
with open("error.txt", "w", encoding="utf-8") as file:
    db = get_spacy_doc(file, train)
    db.to_disk("train_data.spacy")

    db = get_spacy_doc(file, test)
    db.to_disk("test_data.spacy")

100%|██████████| 60/60 [00:00<00:00, 123.37it/s]


In [ ]:
len(db.tokens)

60

In [ ]:
!python -m spacy train /content/CV-Parsing-using-Spacy-3/data/training/config.cfg \
  --output /content/CV-Parsing-using-Spacy-3/output \
  --paths.train /content/train_data.spacy \
  --paths.dev /content/test_data.spacy \
  --gpu-id 0


ℹ Saving to output directory:
/content/CV-Parsing-using-Spacy-3/output
ℹ Using GPU: 0
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/spacy/__main__.py", line 4, in <module>
    setup_cli()
  File "/usr/local/lib/python3.12/dist-packages/spacy/cli/_util.py", line 87, in setup_cli
    command(prog_name=COMMAND)
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1442, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/typer/core.py", line 757, in main
    return _main(
           ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/typer/core.py", line 195, in _main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1830, in invoke
    return _process_result(sub_ctx.command.invo